In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FixDatabase") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.sql.catalog.hive", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hive.catalog-impl", "org.apache.iceberg.hive.HiveCatalog") \
    .config("spark.sql.catalog.hive.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.hive.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.defaultCatalog", "hive") \
    .getOrCreate()

25/09/20 12:06:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
from minio import Minio
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)
bucket = "warehouse"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [3]:
spark.sql("""
select count(*) from test_db.test_table
""").show()

25/09/20 12:06:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------+
|count(1)|
+--------+
|       9|
+--------+



In [4]:
# Create database with explicit location
spark.sql("CREATE DATABASE IF NOT EXISTS test_db LOCATION 's3a://warehouse/test_db'")

DataFrame[]

In [5]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+



In [6]:
# Create a sample Iceberg table
spark.sql("""
CREATE TABLE IF NOT EXISTS test_db.test_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
)
USING iceberg
LOCATION 's3a://warehouse/test_db/test_table'
""")

DataFrame[]

In [7]:
# Insert sample data
spark.sql("""
INSERT INTO test_db.test_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP),
    (3, 'Charlie', CURRENT_TIMESTAMP)
""")

DataFrame[]

In [8]:
spark.sql("""
select count(*) from test_db.test_table
""").show()

+--------+
|count(1)|
+--------+
|      12|
+--------+



In [9]:
# Query the table
result = spark.sql("SELECT * FROM test_db.test_table")
result.show()

+---+-------+--------------------+
| id|   name|          created_at|
+---+-------+--------------------+
|  1|  Alice|2025-09-20 11:55:...|
|  2|    Bob|2025-09-20 11:55:...|
|  3|Charlie|2025-09-20 11:55:...|
|  1|  Alice|2025-09-20 11:52:...|
|  1|  Alice|2025-09-20 12:07:...|
|  1|  Alice|2025-09-20 12:00:...|
|  2|    Bob|2025-09-20 11:52:...|
|  3|Charlie|2025-09-20 11:52:...|
|  2|    Bob|2025-09-20 12:07:...|
|  3|Charlie|2025-09-20 12:07:...|
|  2|    Bob|2025-09-20 12:00:...|
|  3|Charlie|2025-09-20 12:00:...|
+---+-------+--------------------+



In [10]:
# Verify table metadata in Hive Metastore
tables = spark.sql("SHOW TABLES IN test_db")
tables.show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  test_db|test_table|      false|
+---------+----------+-----------+



In [11]:
spark.sql("SELECT * FROM test_db.test_table.snapshots").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-09-20 11:52:...|5906623575764873981|               NULL|   append|s3a://warehouse/t...|{spark.app.id -> ...|
|2025-09-20 11:55:...|7751092166435894797|5906623575764873981|   append|s3a://warehouse/t...|{spark.app.id -> ...|
|2025-09-20 12:00:...|5619447774994201361|7751092166435894797|   append|s3a://warehouse/t...|{spark.app.id -> ...|
|2025-09-20 12:07:...|  26592669950855846|5619447774994201361|   append|s3a://warehouse/t...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [12]:
spark.sql("SHOW DATABASES IN hive").show()

+---------+
|namespace|
+---------+
|  default|
|  test_db|
+---------+

